# P5 — Serving de Modelos con API REST

**RA/CE**: RA3d (soluciones basadas en IA)
**Fase**: F5 — Serving y APIs
**Teoría asociada**: `01-teoria/06_serving_apis.md`

---

## Objetivos de Aprendizaje

1. Servir un modelo entrenado como API REST con FastAPI
2. Implementar autenticación con API key
3. Añadir rate limiting para proteger la API
4. Versionar endpoints para migración gradual de clientes
5. Probar la API con `httpx` desde el notebook
6. Evaluar cómo el serving profesional constituye una solución basada en IA (RA3d)

## Prerrequisitos

- [ ] FastAPI básico (visto en UD6: `103_fastapi_serving_modelos.ipynb`)
- [ ] Finalizada la práctica P4 (tienes un modelo registrado en MLflow)
- [ ] FastAPI y uvicorn instalados: `pip install fastapi uvicorn httpx slowapi`

## Contexto

En P4 creaste un pipeline orquestado con Prefect que entrena modelos y los
registra en MLflow. Ahora vamos a **servir ese modelo como una API profesional**:
con autenticación, rate limiting, versionado y documentación automática.

---
## Parte 1: Verifica tu entorno

In [ ]:
import sys

HAS_DEPS = True
missing = []

try:
    from fastapi import FastAPI
    import uvicorn
    print('✅ FastAPI + Uvicorn OK')
except ImportError as e:
    HAS_DEPS = False
    missing.append('fastapi uvicorn')
    print('❌ Missing: fastapi/uvicorn')

try:
    from slowapi import Limiter
    print('✅ SlowAPI OK')
except ImportError:
    print('⚠️ SlowAPI not installed. Optional for rate limiting.')

try:
    import httpx
    print('✅ HTTPX OK')
except ImportError:
    print('⚠️ HTTPX not installed. Install: pip install httpx')

try:
    import mlflow
    from pydantic import BaseModel
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    print('✅ ML dependencies OK')
except ImportError as e:
    HAS_DEPS = False
    mod = str(e).split("'")[1] if "'" in str(e) else str(e)
    missing.append(mod)
    print(f'❌ Missing: {mod}')

if not HAS_DEPS:
    print(f'\nInstall: pip install {" ".join(missing)}')
else:
    print('\n✅ Environment ready')
    print(f'Python: {sys.version[:6]}')

---
## Parte 2: Entrenar y registrar un modelo de referencia

Si no tienes un modelo registrado de prácticas anteriores, entrenamos uno
ahora y lo registramos en MLflow.

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os, json, pickle

# Crear datos si no existen
if not os.path.exists("02-ejemplos/pipeline_completo/data/raw/tickets.csv"):
    os.makedirs("02-ejemplos/pipeline_completo/data/raw", exist_ok=True)
    os.makedirs("02-ejemplos/pipeline_completo/data/processed", exist_ok=True)
    
    np.random.seed(42)
    categorias = ['red', 'system', 'database', 'security', 'application']
    descripciones = [
        'El servidor de producción no responde',
        'Error de conexión a la base de datos PostgreSQL',
        'Ataque de fuerza bruta detectado en el firewall',
        'La aplicación web tarda más de 30 segundos en cargar',
        'El certificado SSL ha expirado',
    ] * 20
    
    data = []
    for desc in descripciones:
        data.append({'description': desc, 'category': np.random.choice(categorias)})
    
    df = pd.DataFrame(data)
    df.to_csv("02-ejemplos/pipeline_completo/data/raw/tickets.csv", index=False)
else:
    df = pd.read_csv("02-ejemplos/pipeline_completo/data/raw/tickets.csv")

# Entrenar modelo
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['description'])
y = df['category']

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X, y)

# Guardar localmente para la API
os.makedirs("models", exist_ok=True)
with open("models/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
with open("models/classifier.pkl", "wb") as f:
    pickle.dump(model, f)

print(f'✅ Modelo guardado en models/')
print(f'Categorías: {list(model.classes_)}')

---
## Parte 3: Crear la API FastAPI

Vamos a construir una API profesional con:
- Modelos Pydantic para validación de entrada/salida
- Autenticación mediante API key
- Rate limiting con slowapi
- Endpoints versionados (`/v1/predict`, `/v2/predict`)

In [ ]:
from fastapi import FastAPI, HTTPException, Security
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel, Field
from typing import Dict, Optional, List
import pickle

# ---- Configuración ----
API_KEYS = {
    "sk-soporte-v1": "soporte",
    "sk-movil-v1": "app-movil",
    "sk-admin-v1": "admin",
}

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

# ---- Modelos Pydantic ----
class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=10000,
                      description="Texto de la incidencia a clasificar")
    top_k: Optional[int] = Field(3, ge=1, le=10,
                                  description="Número de categorías a devolver")

class PredictResponse(BaseModel):
    predicted_class: str
    confidence: float = Field(..., ge=0.0, le=1.0)
    probabilities: Dict[str, float]
    model_version: str

class ErrorResponse(BaseModel):
    error: str
    detail: str

# ---- Crear la aplicación ----
app = FastAPI(
    title="Ticket Classifier API",
    description="API de clasificación de incidencias técnicas. "
                "Parte del stack convergente de UD7.",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)

# ---- Rate Limiting ----
try:
    from slowapi import Limiter, _rate_limit_exceeded_handler
    from slowapi.util import get_remote_address
    from slowapi.errors import RateLimitExceeded
    
    limiter = Limiter(key_func=get_remote_address)
    app.state.limiter = limiter
    app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)
    print('✅ Rate limiting configurado')
except ImportError:
    limiter = None
    print('⚠️ SlowAPI no disponible, saltando rate limiting')

# ---- Cargar modelos ----
@app.on_event("startup")
async def load_models():
    """Carga los modelos al iniciar la API."""
    global vectorizer_v1, model_v1
    global vectorizer_v2, model_v2
    
    with open("models/vectorizer.pkl", "rb") as f:
        vectorizer_v1 = pickle.load(f)
    with open("models/classifier.pkl", "rb") as f:
        model_v1 = pickle.load(f)
    
    # v2 usa el mismo modelo (simula una versión mejorada)
    vectorizer_v2 = vectorizer_v1
    model_v2 = model_v1
    
    print('✅ Modelos cargados')

# ---- Helper: verificar API key ----
def verify_api_key(api_key: str = Security(api_key_header)) -> str:
    """Verifica la API key y devuelve el nombre del cliente."""
    if api_key is None or api_key not in API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="API Key inválida. Incluye X-API-Key en los headers.",
            headers={"WWW-Authenticate": "API Key"},
        )
    return API_KEYS[api_key]

print('✅ API definida. Modelos Pydantic, auth y rate limiting listos.')

### 3.1 Endpoints de la API

Definimos los endpoints de inferencia, incluyendo versionado.

In [ ]:
# ---- Endpoint raíz ----
@app.get("/")
async def root():
    return {
        "api": "Ticket Classifier API",
        "version": "1.0.0",
        "endpoints": ["/predict", "/v1/predict", "/v2/predict"],
        "docs": "/docs",
    }

# ---- Endpoint de salud ----
@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": True}

# ---- Helper de predicción ----
def predict_text(text: str, vectorizer, model, top_k: int = 3):
    """Ejecuta la predicción y devuelve resultados estructurados."""
    X = vectorizer.transform([text])
    pred_class = model.predict(X)[0]
    probs = model.predict_proba(X)[0]
    
    top_indices = np.argsort(probs)[-top_k:][::-1]
    top_probs = {
        model.classes_[i]: float(probs[i])
        for i in top_indices
    }
    
    return PredictResponse(
        predicted_class=pred_class,
        confidence=float(probs[model.classes_.tolist().index(pred_class)]),
        probabilities=top_probs,
        model_version="v1"
    )

# ---- Endpoint /predict (v1, por defecto) ----
@app.post("/predict", response_model=PredictResponse)
async def predict(
    request: PredictRequest,
    client: str = Security(verify_api_key)
):
    """Clasifica un texto de incidencia (versión estable)."""
    return predict_text(
        request.text, vectorizer_v1, model_v1, request.top_k
    )

# ---- Endpoint /v1/predict ----
@app.post("/v1/predict", response_model=PredictResponse)
async def predict_v1(
    request: PredictRequest,
    client: str = Security(verify_api_key)
):
    """Endpoint v1: modelo LogisticRegression estable."""
    return predict_text(
        request.text, vectorizer_v1, model_v1, request.top_k
    )

# ---- Endpoint /v2/predict ----
@app.post("/v2/predict", response_model=PredictResponse)
async def predict_v2(
    request: PredictRequest,
    client: str = Security(verify_api_key)
):
    """Endpoint v2: modelo mejorado (nuevas categorías, más preciso)."""
    return predict_text(
        request.text, vectorizer_v2, model_v2, request.top_k
    )

print('✅ Todos los endpoints definidos')

---
## Parte 4: Iniciar la API y probar los endpoints

Iniciamos el servidor en segundo plano y probamos los endpoints con `httpx`.

In [ ]:
import threading, time, signal

# Iniciar servidor en un hilo separado
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="error")
server = uvicorn.Server(config=config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(2)  # Esperar a que el servidor arranque

print('✅ Servidor FastAPI iniciado en http://127.0.0.1:8000')
print(f'   Documentación: http://127.0.0.1:8000/docs')

### 4.1 Probar endpoint raíz

In [ ]:
import httpx

response = httpx.get("http://127.0.0.1:8000/")
print(json.dumps(response.json(), indent=2))

### 4.2 Probar predicción con autenticación correcta

In [ ]:
# Predicción con API key válida
response = httpx.post(
    "http://127.0.0.1:8000/predict",
    headers={"X-API-Key": "sk-soporte-v1"},
    json={"text": "El servidor de producción no responde desde las 14:00", "top_k": 3}
)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

### 4.3 Probar error de autenticación (sin API key)

In [ ]:
# Sin API key → debe devolver 401
response = httpx.post(
    "http://127.0.0.1:8000/predict",
    json={"text": "test"}
)
print(f"Status: {response.status_code} (esperado: 401)")
print(json.dumps(response.json(), indent=2))

### 4.4 Probar endpoints versionados

In [ ]:
# Endpoint v1
resp1 = httpx.post(
    "http://127.0.0.1:8000/v1/predict",
    headers={"X-API-Key": "sk-soporte-v1"},
    json={"text": "Error 500 en el endpoint de pagos"}
)
print("v1:", resp1.json()["predicted_class"], f"(conf: {resp1.json()['confidence']:.3f})")

# Endpoint v2
resp2 = httpx.post(
    "http://127.0.0.1:8000/v2/predict",
    headers={"X-API-Key": "sk-soporte-v1"},
    json={"text": "Error 500 en el endpoint de pagos"}
)
print("v2:", resp2.json()["predicted_class"], f"(conf: {resp2.json()['confidence']:.3f})")

### 4.5 Probar rate limiting

El rate limiter está configurado a 10 peticiones/minuto para el endpoint `/predict`.
Vamos a enviar más de 10 peticiones rápidas y ver cómo responde.

In [ ]:
# Probar rate limiting (se envía rápido para superar el límite)
codes = []
for i in range(15):
    r = httpx.post(
        "http://127.0.0.1:8000/predict",
        headers={"X-API-Key": "sk-soporte-v1"},
        json={"text": f"Prueba {i}"}
    )
    codes.append(r.status_code)
    if r.status_code == 429:
        print(f"Petición {i+1}: 429 Too Many Requests 🚫")
        break
    elif i == 14:
        print(f"Última petición: {r.status_code}")

print(f"\nCódigos de estado: {codes[:8]}...")

---
## Parte 5: Conexión con el Stack Convergente

La API que acabas de construir se conecta con el resto del stack:

| Componente | Conexión |
|------------|----------|
| **F3 MLflow** | El modelo se carga desde el Model Registry (o desde archivo en esta práctica) |
| **F4 Prefect** | Cuando Prefect re-entrena un modelo, promueve la nueva versión a Production en MLflow |
| **F6 Agentes** | Los agentes consumen esta API para clasificar incidencias antes de procesarlas |
| **F7 Evidently** | Las predicciones de la API se monitorizan para detectar drift |

**RA3d en acción**: Una solución basada en IA no es solo el modelo —es el sistema
completo que lo sirve (API), lo protege (auth, rate limiting), lo versiona
(endpoints v1/v2) y lo documenta (Swagger).

---
## Parte 6: Ejercicios

### Ejercicio 1: Endpoint de predicción batch

Añade un endpoint `/v2/predict_batch` que acepte una lista de textos y
devuelva predicciones para todos ellos. Usa el modelo v2.

In [ ]:
# TODO: Implementa el endpoint de predicción batch

class BatchPredictRequest(BaseModel):
    texts: List[str] = Field(..., min_length=1, max_length=100)

class BatchPredictResponse(BaseModel):
    predictions: List[PredictResponse]
    total: int

# Tu código aquí

### Ejercicio 2: Logging de peticiones

Añade logging a la API para registrar cada petición: cliente, texto
(truncado a 50 caracteres), predicción y tiempo de respuesta.

In [ ]:
# TODO: Implementa logging de peticiones
# Pista: usa dependencia de FastAPI o middleware
# @app.middleware("http")
# async def log_requests(request, call_next):
#     ...

### Ejercicio 3: Análisis RA3d

Responde: ¿Qué elementos de esta API la convierten en una "solución basada en IA"
según RA3d? ¿Qué mejoras añadirías para que sea una solución profesional completa?

In [ ]:
# Escribe tu análisis aquí

---
## Resumen de la Práctica

✅ Has creado una API FastAPI profesional para servir un modelo ML
✅ Has implementado autenticación con API key
✅ Has añadido rate limiting para proteger la API
✅ Has versionado endpoints para migración gradual de clientes
✅ Has probado la API con httpx, incluyendo casos de error
✅ Has analizado cómo el serving profesional materializa RA3d

**Próxima práctica (P6)**: Construirás un sistema multi-agente con CrewAI
que consume esta API para clasificar y procesar incidencias.